In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
pd.set_option('display.max_columns', 100)

In [3]:
pd.set_option('display.width', 140)

In [4]:
# Path to Prepared Data folder
BASE_PREPARED = Path(r"C:\Users\Pocah\OneDrive\Desktop\bootcamp\python\Prepared Data")

In [5]:
INPUT_FILE = BASE_PREPARED / "orders_products_combined.pkl"

In [6]:
OUTPUT_FILE = BASE_PREPARED / "orders_products_combined_updated.pkl"

In [7]:
print("Input file:", INPUT_FILE)

Input file: C:\Users\Pocah\OneDrive\Desktop\bootcamp\python\Prepared Data\orders_products_combined.pkl


In [8]:
print("Output file:",OUTPUT_FILE)

Output file: C:\Users\Pocah\OneDrive\Desktop\bootcamp\python\Prepared Data\orders_products_combined_updated.pkl


## Load Data — I’ll load the dataset and check the shape, columns, and a few rows.

In [9]:
# Load dataset
if not INPUT_FILE.exists():
    raise FileNotFoundError(f"Could not find: {INPUT_FILE}")

df = pd.read_pickle(INPUT_FILE)
print("Shape:", df.shape)
print("Columns:", list(df.columns)[:25])
df.head(3)

Shape: (32434489, 11)
Columns: ['order_id', 'user_id', 'eval_set', 'order_number', 'order_day_of_week', 'order_hour_of_day', 'days_since_prior_order', 'product_id', 'add_to_cart_order', 'reordered', '_merge']


,order_id,user_id,eval_set,order_number,order_day_of_week,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,_merge
0,2539329,1,prior,1,2,8,0.0,196,1,0,both
1,2539329,1,prior,1,2,8,0.0,14084,2,0,both
2,2539329,1,prior,1,2,8,0.0,12427,3,0,both


## Derivation A — busiest_days

In [10]:
# BEFORE: counts per day of week
busiest_days_counts = df["order_day_of_week"].value_counts().sort_index()
print("Orders per day_of_week (BEFORE):")
display(busiest_days_counts)

Orders per day_of_week (BEFORE):


order_day_of_week
0    6209666
1    5665856
2    4217798
3    3844117
4    3787215
5    4209533
6    4500304
Name: count, dtype: int64

In [11]:
counts_desc = busiest_days_counts.sort_values(ascending=False)
top_levels = sorted(counts_desc.unique(), reverse=True)[:2]
bottom_levels = sorted(busiest_days_counts.unique())[:2]

In [12]:
top_days = counts_desc[counts_desc.isin(top_levels)].index.tolist()
bottom_days = busiest_days_counts[busiest_days_counts.isin(bottom_levels)].index.tolist()

In [13]:
def label_busiest_days(day):
    if day in top_days:
        return "Busiest days"
    elif day in bottom_days:
        return "Slowest days"
    return "Average days"

In [14]:
# AFTER: apply & show results
df["busiest_days"] = df["order_day_of_week"].apply(label_busiest_days)
print("\nNew column 'busiest_days' created. Distribution:")
df["busiest_days"].value_counts()


New column 'busiest_days' created. Distribution:


busiest_days
Average days    12927635
Busiest days    11875522
Slowest days     7631332
Name: count, dtype: int64

##Derivation — busiest_period_of_day

In [15]:
#counts per hour
hourly_counts = df["order_hour_of_day"].value_counts().sort_index()
print("Orders per hour (BEFORE):")
display(hourly_counts)

Orders per hour (BEFORE):


order_hour_of_day
0      218948
1      115786
2       69434
3       51321
4       53283
5       88062
6      290795
7      891937
8     1719973
9     2456713
10    2764426
11    2738582
12    2620847
13    2663292
14    2691548
15    2664533
16    2537458
17    2089465
18    1637923
19    1259401
20     977038
21     796370
22     634734
23     402620
Name: count, dtype: int64

In [16]:
# Map_0–6 to_weekday_names
day_map = {
    0: "Sunday",
    1: "Monday",
    2: "Tuesday",
    3: "Wednesday",
    4: "Thursday",
    5: "Friday",
    6: "Saturday"
}

# Create_a_new_column_with_weekday_names
df["order_day_name"] = df["order_day_of_week"].map(day_map)

# Check distribution by day name
busiest_days = df["order_day_name"].value_counts().sort_index()
print("Orders per weekday:")
display(busiest_days)


Orders per weekday:


order_day_name
Friday       4209533
Monday       5665856
Saturday     4500304
Sunday       6209666
Thursday     3787215
Tuesday      4217798
Wednesday    3844117
Name: count, dtype: int64

In [17]:
#thresholds
most_threshold = hourly_counts.quantile(0.75)
fewest_threshold = hourly_counts.quantile(0.25)

In [18]:
def label_period(hour):
    count = hourly_counts.get(hour, 0)  # safe lookup
    if count >= most_threshold:
        return "Most orders"
    elif count <= fewest_threshold:
        return "Fewest orders"
    else:
        return "Average orders"

In [19]:
#apply & show results
df["busiest_period_of_day"] = df["order_hour_of_day"].apply(label_period)
print("\nNew column 'busiest_period_of_day' created. Distribution:")


New column 'busiest_period_of_day' created. Distribution:


In [20]:
df["busiest_period_of_day"].value_counts()

busiest_period_of_day
Most orders       16143228
Average orders    15694427
Fewest orders       596834
Name: count, dtype: int64

In [21]:
# Price range (if prices exist)
if "prices" in df.columns:
    df["price_range"] = pd.cut(df["prices"], bins=[-np.inf, 5, 15, np.inf],
                               labels=["Low", "Mid", "High"])
    print("Price range created:")
    display(df["price_range"].value_counts())

In [22]:
#Work Summary
cols = [c for c in ["order_day_of_week", "busiest_days",
                    "order_hour_of_day", "busiest_period_of_day",
                    "price_range", "recency_band"] if c in df.columns]
df[cols].head(12)

,order_day_of_week,busiest_days,order_hour_of_day,busiest_period_of_day
0,2,Average days,8,Average orders
1,2,Average days,8,Average orders
2,2,Average days,8,Average orders
3,2,Average days,8,Average orders
4,2,Average days,8,Average orders
5,3,Slowest days,7,Average orders
6,3,Slowest days,7,Average orders
7,3,Slowest days,7,Average orders
8,3,Slowest days,7,Average orders
9,3,Slowest days,7,Average orders


##Save the updated dataset 

In [23]:
df.to_pickle(OUTPUT_FILE)
print(f"Saved updated dataframe to: {OUTPUT_FILE}")

Saved updated dataframe to: C:\Users\Pocah\OneDrive\Desktop\bootcamp\python\Prepared Data\orders_products_combined_updated.pkl
